# Isla-SNN: Instruction Fine-Tuning 💬

Use this notebook on **Google Colab (T4/L4 GPU)** to Fine-Tune the Base SNN model into an Assistant.
It downloads the source code, mounts your Google Drive to load the Pre-Trained model, maps the dataset, and applies **Prompt Masking** (`-100` Loss) automatically.

In [ ]:
!git clone https://github.com/Heitorkk2/Isla-SNN.git
%cd Isla-SNN
!pip install -q -e ".[dev]"

## 1. Load Base Model
Change `MODEL_PATH` to wherever your model is saved in Drive.

In [ ]:
MODEL_PATH = "/content/isla-SNN/outputs/final"
device = "cuda" if torch.cuda.is_available() else "cpu"

model, tokenizer = isla.load_model(MODEL_PATH, device=device)
model.train()
print(f"[ISLA] Base model loaded with {model.count_params():,} params.")

## 2. Dataset Preparation
We are using `llm-wizard/dolly-15k-instruction-alpaca-format`.
We merge its columns to standard Alpaca Format and append the tokenizer's Stop Token (`EOS`).

In [ ]:
from datasets import load_dataset

# 1. Download raw dataset
raw_ds = load_dataset("llm-wizard/dolly-15k-instruction-alpaca-format", split="train")
eos = tokenizer.eos_token

# 2. Format into a single 'text' column ending with the tokenizer's EOS
def format_alpaca(example):
    text = f"### Instruction:\n{example['instruction']}\n\n"
    if example.get('input'):
        text += f"### Input:\n{example['input']}\n\n"
    text += f"### Response:\n{example['output']}{eos}"
    return {"text": text}

formatted_ds = raw_ds.map(format_alpaca, remove_columns=raw_ds.column_names)

# 3. Save locally
import os; os.makedirs("./data", exist_ok=True)
formatted_ds.to_json("./data/dolly_formatted.jsonl")
print("\nDataset Sample:")
print(formatted_ds[0]["text"])

## 3. Configuration & Tokenization

In [ ]:
train_config = isla.TrainConfig(
    lr=5e-5,  # Lower LR for fine-tuning
    num_epochs=1,
    batch_size=16,
    gradient_accumulation_steps=2,
    bf16=True if torch.cuda.is_bf16_supported() else False,
    fp16=True if not torch.cuda.is_bf16_supported() else False,
    log_every=10,
    eval_every=500,
    checkpoint=isla.CheckpointConfig(
        output_dir="/content/isla-chat-finetuned",
        save_every=500,
    ),
    wandb=isla.WandbConfig(enabled=False)
)

data_config = isla.DataConfig(
    dataset_path="./data/dolly_formatted.jsonl",
    tokenizer_name=model.config.tokenizer_name,
    max_seq_len=512,
    pack_sequences=True,
    is_finetune=True,                       # Turn on -100 Prompt Masking!
    response_template="### Response:\n",    # Loss calculates after this token
)

In [ ]:
from isla.data import load_isla_dataset, create_dataloader

dataset = load_isla_dataset(
    data_config.dataset_path, 
    tokenizer, 
    data_config.max_seq_len, 
    pack=data_config.pack_sequences,
    is_finetune=data_config.is_finetune,
    response_template=data_config.response_template
)

train_dl = create_dataloader(dataset["train"], train_config.batch_size, shuffle=True)
val_dl = create_dataloader(dataset["validation"], train_config.batch_size, shuffle=False) if "validation" in dataset else None

print("Datset tokenized and masked successfully!")

## 4. Train

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

trainer = isla.IslaTrainer(model, train_config, model.config)
trainer.train(train_dl, val_dl)



## 4. Evaluation (Chat Test)

Let's see if it learned to stop rambling!

In [ ]:
model.eval()

prompt = ""
print(f"\n{prompt}")

with torch.no_grad(), torch.autocast(device_type="cuda" if "cuda" in device else "cpu", dtype=torch.bfloat16):
    for word in isla.generate_stream(
        model,
        tokenizer,
        prompt,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        device=device
    ):
        print(word, end="", flush=True)
print("\n--- [END] ---")